## Regression (Fatwa's popularity)

In [1]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import ElasticNet, Ridge, Lasso
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error



In [2]:
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression



In [22]:
#df_k = pd.read_csv('/Users/market/Desktop/thesis_project/data/df_new2.csv')

df_k = pd.read_csv("/Users/market/Desktop/thesis_project/data/fatwas_with_probabilities.csv")


In [ ]:
df_k

In [23]:
#added title length as a feature
df_k['title_length'] = df_k['title'].str.len()

In [24]:
#Removing stopwords

file_path = "/Users/market/Desktop/thesis_project/stopwords/kazakh_stopwords.txt"

with open(file_path, "r", encoding="utf-8") as file:
    kz_stopwords = []
    for line in file:
        word = line.strip().lower()              # убрать пробелы и \n, привести к lower
        word = word.replace('"', '').replace("'", '')  # убрать кавычки
        word = re.sub(r'\s+', ' ', word)         # заменить несколько пробелов на один
        if word:                                 # пропустить пустые строки
            kz_stopwords.append(word)

In [25]:
#removing stopwords and extra spaces from titles
df_k['title'] = df_k['title'].apply(lambda x: re.sub(r'\s+', ' ', ' '.join([word for word in str(x).split() if word.lower() not in kz_stopwords])).strip())


In [ ]:
#Надо теперь "лемматизировать" - но вручную - так как автоматические инструменты для казахского не очень работают 
# - нужно просто удалить все окончания и привести к базовой форме - например, "дұға", "дұғалар", "дұғаны" - это все одна и та же лемма "дұға" - нужно удалить окончания и оставить только корень слова, который будет использоваться в качестве признака


In [26]:
#Подход ниже учитывает факта, что в казахском языке может быть несколько слоев суффиксов,
#  и удаляет их итеративно, пока не останется корень слова или слово не станет слишком коротким. 
# Это позволяет более эффективно лемматизировать слова и уменьшить количество признаков в модели, 
# сохраняя при этом смысловую нагрузку слов.


def deep_clean_kazakh(text):
    if not isinstance(text, str):
        return ""
    
    # Приводим к нижнему регистру и убираем пунктуацию, чтобы не мешала границам слов
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    
    # Список окончаний: от длинных к коротким (важно для корректного срабатывания)
    # Добавил также специфические суффиксы принадлежности (ым, ың и т.д.)
    pattern = r'(лар|лер|дар|дер|тар|тер|дың|дің|тың|тің|ның|нің|лық|лік|дық|дік|тық|тік|да|де|та|те|қа|ке|ға|ге|ны|ні|ты|ті|дан|ден|тан|тен|нан|нен|мен|бен|пен|ым|ім|ың|ің|ы|і|ын)\b'
    
    words = text.split()
    cleaned_words = []
    
    for word in words:
        # Прогоняем слово через цикл удаления суффиксов (до 3-х слоев)
        for _ in range(3):
            new_word = re.sub(pattern, '', word)
            if new_word == word or len(new_word) < 3: # Останавливаемся, если корень стал слишком коротким
                break
            word = new_word
        cleaned_words.append(word)
    
    return " ".join(cleaned_words)

# 2. Применяем к датафрейму
df_k['title_cleaned'] = df_k['title'].apply(deep_clean_kazakh)

# Посмотрим, что получилось
print("Пример очистки:")
print(df_k[['title', 'title_cleaned']].head(10))

Пример очистки:
                                               title  \
0          діни тұрғыда музыкалық аспаптарды сатудың   
1                      адамды өсектесе кешірім сұрау   
2          елдерде шарап сынды заттарды сатуға пәтуа   
3  әйелдің күйеуінің тегін фамилиясын алуы күпірл...   
4  бастау тәкбірінде аллаһу әкбар дегеннің орнына...   
5               күйеуім әйелім емессің талақ талаққа   
6        амандасқанда сәлеміңді алмаса періште алады   
7                                           құрбанға   
8                                     хабашит кімдер   
9                        жұма күні мал союға естідім   

                                       title_cleaned  
0                    діни тұрғ музыка аспаптард сату  
1                         адамд өсектесе кешір сұрау  
2                елдер шарап сынд заттард сату пәтуа  
3    әйел күйеу тегін фамилияс алу күпір дегенд оқыд  
4  бастау тәкбірін аллаһу әкбар деген орнына қолд...  
5                       күйеу әйел ем

## 1. Target и лог-трансформация
Просмотры почти всегда heavy-tailed, поэтому:

In [27]:
y = np.log1p(df_k['views'])

df_k['log_days_passed'] = np.log1p(df_k['days_passed'])


## 2. Предикторы

In [10]:
X = df_k[['title_cleaned', 'hijri_month', 'log_days_passed', 'title_length']]

## 3. Препроцессинг
Word-based TF-IDF and Character-based TF-IDF

In [11]:
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9,
    max_features=20000
)

In [12]:
char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3,5),
    min_df=5,
    max_df=0.9,
    max_features=30000
)


### Числовые признаки


In [13]:
numeric_transformer = Pipeline(steps=[
    ('log', StandardScaler())
])

### One-hot кодирование исламских месяцев


In [14]:
month_encoder = OneHotEncoder(
    categories='auto',
    drop='first',
    sparse_output=True
)

## 4. ColumnTransformer — сердце всей архитектуры

Разбор каждой «двери» (трансформатора):

('word', word_vectorizer, 'title')

Берет колонку 'title' (твои фетвы) и превращает слова в числа (вектора). Скорее всего, здесь используется что-то вроде TF-IDF или CountVectorizer. Это те самые word-based features.

('char', char_vectorizer, 'title')

Снова берет ту же колонку 'title', но анализирует её на уровне символов (character-based features). Это поможет модели уловить специфические казахские суффиксы и окончания.

('month', month_encoder, ['hijri_month'])

Берет колонку с месяцами по Хиджре и применяет тот самый OneHotEncoder, который мы обсуждали выше. Это важно, так как религиозный календарь напрямую влияет на интерес к фетвам.

('days', StandardScaler(), ['days_passed'])

StandardScaler — это очень важная штука для нейросетей. Она приводит числа (сколько дней прошло с публикации) к единому масштабу (обычно от -3 до 3).

Зачем: Если у тебя количество дней — 1000, а в других колонках только 0 и 1, нейросеть «сойдет с ума» от таких больших чисел и будет обращать внимание только на них. Скейлер делает все данные «равноправными».

Без этого инструмента тебе пришлось бы вручную склеивать таблицы, следить, чтобы строки не перепутались, и отдельно сохранять параметры для каждой колонки. ColumnTransformer делает это одной командой preprocessor.fit_transform(X).


In [15]:
preprocessor_word = ColumnTransformer(
    transformers=[
        ('word', word_vectorizer, 'title_cleaned'),
        ('month', month_encoder, ['hijri_month']),
        ('days', StandardScaler(), ['log_days_passed']),
        ('length', StandardScaler(), ['title_length'])
    ],
    remainder='drop',
    sparse_threshold=0.3
)

preprocessor_char = ColumnTransformer(
    transformers=[
        ('char', char_vectorizer, 'title_cleaned'),
        ('month', month_encoder, ['hijri_month']),
        ('days', StandardScaler(), ['log_days_passed']),
        ('length', StandardScaler(), ['title_length'])
    ],
    remainder='drop',
    sparse_threshold=0.3
)



In [16]:
def build_models(preprocessor):
    return {
        'OLS_SVD': Pipeline([
            ('preprocess', preprocessor),
            ('svd', TruncatedSVD(n_components=300, random_state=42)),
            ('model', LinearRegression())
        ]),
        'Ridge': Pipeline([
            ('preprocess', preprocessor),
            ('model', Ridge(alpha=1.0))
        ]),
        'Lasso': Pipeline([
            ('preprocess', preprocessor),
            ('model', Lasso(alpha=0.001, max_iter=5000))
        ]),
        'ElasticNet': Pipeline([
            ('preprocess', preprocessor),
            ('model', ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))
        ])
    }


In [17]:
model_groups = {
    'word_only': build_models(preprocessor_word),
    'char_only': build_models(preprocessor_char)
}


## 5. МОДЕЛИ
Мы сделаем сразу 4  модели:

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import MaxAbsScaler

Мы сделаем:
OLS только на топ-N признаках
или через TruncatedSVD (LSA) → потом OLS

Самый академически чистый вариант:
TF-IDF → SVD → OLS
Это:
-снимает мультиколлинеарность
-делает матрицу плотной
-позволяет нормальную интерпретацию

## 6. Train-test split


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 7. Обучение + сравнение моделей

In [20]:
from sklearn.metrics import root_mean_squared_error


results = []

for group_name, models in model_groups.items():
    for model_name, pipe in models.items():

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        r2 = r2_score(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)

        results.append({
            'group': group_name,
            'model': model_name,
            'R2': r2,
            'RMSE': rmse
        })

        print(f'\n{group_name} | {model_name}')
        print(f'R² = {r2:.4f}')
        print(f'RMSE = {rmse:.4f}')



word_only | OLS_SVD
R² = 0.4462
RMSE = 0.7543

word_only | Ridge
R² = 0.4566
RMSE = 0.7471

word_only | Lasso
R² = 0.4787
RMSE = 0.7318

word_only | ElasticNet
R² = 0.4821
RMSE = 0.7294

char_only | OLS_SVD
R² = 0.4933
RMSE = 0.7215

char_only | Ridge
R² = 0.5134
RMSE = 0.7070

char_only | Lasso
R² = 0.4526
RMSE = 0.7499

char_only | ElasticNet
R² = 0.4791
RMSE = 0.7316


## 8. Как понять, какая модель лучшая

In [21]:
results_df = pd.DataFrame(results)
results_df.sort_values(['group', 'R2'], ascending=[True, False])

,group,model,R2,RMSE
5,char_only,Ridge,0.513433,0.707012
4,char_only,OLS_SVD,0.493258,0.721521
7,char_only,ElasticNet,0.479067,0.731554
6,char_only,Lasso,0.452593,0.749912
3,word_only,ElasticNet,0.482098,0.729422
2,word_only,Lasso,0.478660,0.731839
1,word_only,Ridge,0.456635,0.747138
0,word_only,OLS_SVD,0.446164,0.754303


In [22]:
results_df.pivot_table(
    index='group',
    columns='model',
    values='R2'
)

model,ElasticNet,Lasso,OLS_SVD,Ridge
group,,,,
char_only,0.479067,0.452593,0.493258,0.513433
word_only,0.482098,0.478660,0.446164,0.456635


## 9. Извлечение коэффициентов

Исправить код ниже учитывая что у нас теперь 12 моделей

In [25]:
# 1. Берем конкретную модель
pipeline = model_groups['char_only']['Ridge']

# 2. Обучаем её (без этого коэффициенты .coef_ физически не существуют в памяти)
pipeline.fit(X_train, y_train)

# 3. Теперь достаем данные
feature_names = pipeline.named_steps['preprocess'].get_feature_names_out()
coefs = pipeline.named_steps['model'].coef_.ravel()

# 4. Собираем таблицу
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs
})

# Очистка имен
coef_df['feature'] = coef_df['feature'].str.replace(r'^.*__', '', regex=True)

# Сортировка
coef_df = coef_df.sort_values('coef', ascending=False)



## 10. Топ-20 слов и символных паттернов, повышающих просмотры


In [26]:
# Выходят дубликаты - но по идее это там просто пробелы есть и они тоже учитываются как разные признаки - нужно удалить пробелы и схлопнуть дубликаты, суммируя их веса
# Удаляем пробелы по краям и схлопываем дубликаты, суммируя их веса

coef_df['feature_clean'] = coef_df['feature'].str.strip()
summary_df = coef_df.groupby('feature_clean')['coef'].sum().sort_values(ascending=False)
print(summary_df.head(20))

#дұға дұғ и и тд - это норм, так как это разные characters-based features

feature_clean
дұға    1.710409
дұғ     1.489724
үсін    1.415225
ішу     1.396818
түн     1.345833
жыла    1.345199
жай     1.095403
жыл     1.056877
жеті    1.004370
қаза    0.988458
тар     0.974523
дұ      0.969402
көз     0.913710
қана    0.895627
еті     0.880860
күн     0.848081
айын    0.821707
ақи     0.814968
тара    0.806352
мұр     0.799249
Name: coef, dtype: float64


## 12. Отдельно интерпретируем контролли


### Topic probabilities as predictors

In [30]:
topic_prob_cols = [col for col in df_k.columns if col.startswith('topic_prob_')]

# Собираем все признаки вместе
# Контрольные переменные + Вероятности тем
X = df_k[['hijri_month', 'log_days_passed', 'title_length'] + topic_prob_cols]

# Целевая переменная
y = np.log1p(df_k['views'])

### Обновление ColumnTransformer

In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# Список числовых признаков (включая вероятности тем)
numeric_features = ['log_days_passed', 'title_length'] + topic_prob_cols

# Настройка препроцессора
preprocessor = ColumnTransformer(
    transformers=[
        # Числовые переменные: масштабируем
        ('num', StandardScaler(), numeric_features),
        # Категориальные переменные: OneHotEncoding для месяцев
        ('cat', OneHotEncoder(drop='first', sparse_output=False), ['hijri_month'])
    ]
)

# Ваша функция для создания моделей (остается почти такой же)
def build_models(preprocessor):
    return {
        'OLS_SVD': Pipeline([
            ('preprocess', preprocessor),
            # SVD может быть полезен, если тем слишком много
            ('svd', TruncatedSVD(n_components=min(50, len(X.columns)-1), random_state=42)),
            ('model', LinearRegression())
        ]),
        'Ridge': Pipeline([
            ('preprocess', preprocessor),
            ('model', Ridge(alpha=1.0))
        ]),
        'Lasso': Pipeline([
            ('preprocess', preprocessor),
            ('model', Lasso(alpha=0.001, max_iter=5000))
        ]),
        'ElasticNet': Pipeline([
            ('preprocess', preprocessor),
            ('model', ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))
        ])
    }



In [33]:


# 1. Извлекаем полную таблицу коэффициентов из обученного пайплайна
all_features = numeric_features + cat_feature_names
coef_table = pd.DataFrame({
    'Feature': all_features,
    'Coefficient': ridge_pipeline.named_steps['model'].coef_
})

# 2. Фильтруем: оставляем только вероятности тем
topic_coefficients = coef_table[coef_table['Feature'].str.startswith('topic_prob_')].copy()

# 3. Безопасное извлечение ключевых слов
def get_keywords_safe(feature_name):
    topic_id = int(feature_name.split('_')[-1])
    # Проверяем, существует ли переменная model_bigrams в памяти
    if 'model_bigrams' in globals():
        try:
            return ", ".join([w[0] for w in model_bigrams.get_topic(topic_id)[:4]])
        except:
            return "Keywords not found"
    return "BERTopic model not found in memory"

topic_coefficients['Keywords'] = topic_coefficients['Feature'].apply(get_keywords_safe)

# 4. Сортируем по значению коэффициента
topic_coefficients = topic_coefficients.sort_values(by='Coefficient', ascending=False)

# 5. Вывод только тем
print("Regression Coefficients for Topic Probabilities (Ridge):")
print(topic_coefficients[['Feature', 'Coefficient', 'Keywords']].to_string(index=False))

Regression Coefficients for Topic Probabilities (Ridge):
      Feature  Coefficient                           Keywords
topic_prob_19     0.083532 BERTopic model not found in memory
 topic_prob_6     0.049785 BERTopic model not found in memory
topic_prob_14     0.048188 BERTopic model not found in memory
 topic_prob_0     0.041273 BERTopic model not found in memory
topic_prob_18     0.033319 BERTopic model not found in memory
topic_prob_15     0.026297 BERTopic model not found in memory
topic_prob_24     0.026143 BERTopic model not found in memory
topic_prob_12     0.022855 BERTopic model not found in memory
topic_prob_31     0.019699 BERTopic model not found in memory
 topic_prob_8     0.018558 BERTopic model not found in memory
topic_prob_22     0.018505 BERTopic model not found in memory
 topic_prob_4     0.016746 BERTopic model not found in memory
topic_prob_28     0.016724 BERTopic model not found in memory
topic_prob_23     0.015916 BERTopic model not found in memory
topic_prob_17

In [34]:
topic_coefficients[['Feature', 'Coefficient', 'Keywords']].to_csv('/Users/market/Desktop/thesis_project/data/topic_regression_coefficients.csv', index=False, encoding='utf-8-sig')